# 🔬 Standard IR Benchmarking: SciFact vs. NFCorpus
## BM25 vs. Dense (MiniLM & BGE) vs. Hybrid RRF

This notebook performs an empirical Information Retrieval evaluation across two contrastive standard academic benchmarks from the **BEIR (Benchmarking IR)** suite:

### The Contrastive Benchmark Hypotheses:
1. **SciFact (Scientific Claim Verification)** (~5,183 docs, 300 test queries):
   - **Characteristics**: Highly specific scientific claims (genetics, molecular biology, oncology). Queries feature precise scientific nomenclature, genes, chemicals, and exact terms.
   - **Hypothesis**: Strong **lexical bias**. BM25 is very competitive because specific named entities (`"TNFAIP3"`, `"microRNA-21"`) leave little room for semantic drift.

2. **NFCorpus (NutritionFacts Medical QA)** (~3,633 docs, 323 test queries):
   - **Characteristics**: Non-expert, conversational health and dietary questions mapped against PubMed medical literature abstracts.
   - **Hypothesis**: Severe **vocabulary mismatch / semantic bias**. Layman queries ("how to lower cholesterol naturally") match technical abstracts ("apolipoprotein B attenuation via phytosterols"). Dense embeddings (especially instruction-tuned BGE) will substantially outperform BM25.

3. **Hybrid Reciprocal Rank Fusion (RRF)** ($k=60$):
   - Evaluates whether rank fusion achieves optimal generalized zero-shot robustness across both datasets without score tuning.

In [1]:
# Cell 1: Environment Setup & Library Imports
# !pip install -q rank-bm25 sentence-transformers numpy pandas scikit-learn beir torch

import os
import re
import math
import time
import torch
import numpy as np
import pandas as pd
from typing import List, Dict, Tuple, Any
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
from beir import util
from beir.datasets.data_loader import GenericDataLoader

# Select device (NVIDIA GPU if available, else CPU)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using compute device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Using compute device: cuda
GPU: NVIDIA GeForce RTX 3050 Laptop GPU


## 📦 1. Loading Standard BEIR Benchmarks: SciFact & NFCorpus

We load official test splits directly from the BEIR repository:
- `corpus`: `{doc_id: {"title": "...", "text": "..."}}`
- `queries`: `{query_id: "query text..."}`
- `qrels`: `{query_id: {doc_id: relevance_score}}`

For maximum speed and reproducibility, we provide an option to run either on the full test set or on a quick diagnostic subset (e.g. 50 queries).

In [2]:
# Cell 2: Dataset Downloader and Ingest

os.makedirs("datasets", exist_ok=True)

def load_beir_dataset(dataset_name: str):
    url = f"https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/{dataset_name}.zip"
    data_path = util.download_and_unzip(url, "datasets")
    corpus, queries, qrels = GenericDataLoader(data_folder=data_path).load(split="test")
    return corpus, queries, qrels

print("Loading SciFact dataset...")
scifact_corpus, scifact_queries, scifact_qrels = load_beir_dataset("scifact")
print(f"-> SciFact: {len(scifact_corpus):,} docs | {len(scifact_queries):,} queries | {len(scifact_qrels):,} qrel entries")

print("Loading NFCorpus dataset...")
nfcorpus_corpus, nfcorpus_queries, nfcorpus_qrels = load_beir_dataset("nfcorpus")
print(f"-> NFCorpus: {len(nfcorpus_corpus):,} docs | {len(nfcorpus_queries):,} queries | {len(nfcorpus_qrels):,} qrel entries")

Loading SciFact dataset...


  0%|          | 0/5183 [00:00<?, ?it/s]

-> SciFact: 5,183 docs | 300 queries | 300 qrel entries
Loading NFCorpus dataset...


  0%|          | 0/3633 [00:00<?, ?it/s]

-> NFCorpus: 3,633 docs | 323 queries | 323 qrel entries


## ⚡ 2. Lexical Retrieval Engine (BM25Okapi)

We build a reusable `BM25Retriever` class that:
- Pre-processes documents by concatenating `title` + `text`
- Tokenizes with lowercase alphanumeric regex token extraction
- Builds an inverted index with `BM25Okapi` and returns top-$k$ document IDs with scores.

In [3]:
# Cell 3: BM25 Engine

class BM25Retriever:
    def __init__(self, corpus: Dict[str, Dict[str, str]]):
        self.doc_ids = list(corpus.keys())
        print(f"Building BM25 index over {len(self.doc_ids):,} passages...")
        t0 = time.time()
        # Concatenate title and text
        corpus_texts = [
            f"{corpus[did].get('title', '')} {corpus[did].get('text', '')}".strip()
            for did in self.doc_ids
        ]
        self.tokenized_corpus = [self._tokenize(doc) for doc in corpus_texts]
        self.bm25 = BM25Okapi(self.tokenized_corpus)
        print(f"BM25 index built in {time.time()-t0:.2f}s")

    @staticmethod
    def _tokenize(text: str) -> List[str]:
        return re.findall(r"\b\w+\b", text.lower())

    def retrieve(self, query: str, top_k: int = 100) -> List[Tuple[str, float]]:
        q_tokens = self._tokenize(query)
        scores = self.bm25.get_scores(q_tokens)
        top_indices = np.argsort(scores)[::-1][:top_k]
        return [(self.doc_ids[idx], float(scores[idx])) for idx in top_indices]

## 🧠 3. Dense Retrieval Engine (MiniLM & BAAI/BGE-small)

We load two industry-standard dense bi-encoders:
1. **`all-MiniLM-L6-v2`**: 384-dimensional general-purpose symmetric bi-encoder (~90MB).
2. **`BAAI/bge-small-en-v1.5`**: 384-dimensional asymmetric bi-encoder (~130MB). Requires the official query prefix:
   `"Represent this sentence for searching relevant passages: "`.

Passage embeddings are normalized once (`normalize_embeddings=True`), allowing Cosine Similarity to be computed efficiently via matrix-vector dot product.

In [4]:
# Cell 4: Dense Embedding Retrieval Engine

print("Loading MiniLM bi-encoder...")
model_minilm = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device=device)

print("Loading BGE-small bi-encoder...")
model_bge = SentenceTransformer("BAAI/bge-small-en-v1.5", device=device)
BGE_QUERY_PREFIX = "Represent this sentence for searching relevant passages: "

class DenseRetriever:
    def __init__(
        self,
        corpus: Dict[str, Dict[str, str]],
        model: SentenceTransformer,
        model_name: str,
        query_prefix: str = "",
        batch_size: int = 128
    ):
        self.doc_ids = list(corpus.keys())
        self.model = model
        self.model_name = model_name
        self.query_prefix = query_prefix
        
        print(f"Pre-encoding {len(self.doc_ids):,} passages with {model_name} on {device}...")
        t0 = time.time()
        corpus_texts = [
            f"{corpus[did].get('title', '')} {corpus[did].get('text', '')}".strip()
            for did in self.doc_ids
        ]
        self.doc_embeddings = self.model.encode(
            corpus_texts,
            batch_size=batch_size,
            normalize_embeddings=True,
            show_progress_bar=False
        )
        print(f"-> Encoded in {time.time()-t0:.2f}s | Matrix shape: {self.doc_embeddings.shape}")

    def retrieve(self, query: str, top_k: int = 100) -> List[Tuple[str, float]]:
        formatted_query = f"{self.query_prefix}{query}" if self.query_prefix else query
        query_emb = self.model.encode(
            [formatted_query],
            normalize_embeddings=True,
            show_progress_bar=False
        )[0]
        
        # Dot product of normalized vectors = Cosine Similarity
        scores = np.dot(self.doc_embeddings, query_emb)
        top_indices = np.argsort(scores)[::-1][:top_k]
        return [(self.doc_ids[idx], float(scores[idx])) for idx in top_indices]

Loading MiniLM bi-encoder...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading BGE-small bi-encoder...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

## 🔀 4. Hybrid Retrieval: Reciprocal Rank Fusion (RRF)

Reciprocal Rank Fusion (RRF) solves the score calibration problem between BM25 scores (unbounded floats $\approx 0 - 40$) and Cosine Similarity ([-1, 1]):
$$RRF\_Score(d) = \sum_{m \in M} \frac{1}{k + r_m(d)}$$
Where $r_m(d)$ is the 1-based rank of document $d$ in retriever $m$, and $k$ is the smoothing constant (standard $k = 60$).

In [5]:
# Cell 5: Reciprocal Rank Fusion (RRF)

def reciprocal_rank_fusion(
    ranked_runs: List[List[Tuple[str, float]]],
    k: int = 60,
    top_k: int = 100
) -> List[Tuple[str, float]]:
    """Combines multiple top-k ranked result lists into a single ranked list using RRF."""
    rrf_scores: Dict[str, float] = {}
    for run in ranked_runs:
        for rank, (doc_id, _) in enumerate(run, start=1):
            if doc_id not in rrf_scores:
                rrf_scores[doc_id] = 0.0
            rrf_scores[doc_id] += 1.0 / (k + rank)
            
    sorted_docs = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)
    return sorted_docs[:top_k]

## 📊 5. Standard IR Evaluation Suite

We implement the standard TREC / BEIR evaluation metrics:
- **Hit Rate@k**: 1 if at least one relevant document appears in the top-$k$, else 0.
- **MRR@k (Mean Reciprocal Rank)**: $\frac{1}{\text{rank}}$ of the *first* retrieved relevant document.
- **nDCG@k (Normalized Discounted Cumulative Gain)**: Graded relevance accounting for position discount:
  $$DCG@k = \sum_{i=1}^k \frac{2^{rel_i} - 1}{\log_2(i + 1)}, \quad nDCG@k = \frac{DCG@k}{IDCG@k}$$

In [6]:
# Cell 6: IR Evaluation Metrics

def hit_rate_at_k(retrieved_ids: List[str], ground_truth: Dict[str, int], k: int) -> float:
    for did in retrieved_ids[:k]:
        if ground_truth.get(did, 0) > 0:
            return 1.0
    return 0.0

def mrr_at_k(retrieved_ids: List[str], ground_truth: Dict[str, int], k: int) -> float:
    for rank, did in enumerate(retrieved_ids[:k], start=1):
        if ground_truth.get(did, 0) > 0:
            return 1.0 / rank
    return 0.0

def ndcg_at_k(retrieved_ids: List[str], ground_truth: Dict[str, int], k: int) -> float:
    top_k_ids = retrieved_ids[:k]
    dcg = 0.0
    for rank, did in enumerate(top_k_ids, start=1):
        rel = ground_truth.get(did, 0)
        if rel > 0:
            dcg += (2**rel - 1) / math.log2(rank + 1)
            
    ideal_rels = sorted(ground_truth.values(), reverse=True)[:k]
    idcg = 0.0
    for rank, rel in enumerate(ideal_rels, start=1):
        if rel > 0:
            idcg += (2**rel - 1) / math.log2(rank + 1)
            
    return (dcg / idcg) if idcg > 0.0 else 0.0

def evaluate_dataset(
    benchmark_name: str,
    retrievers: Dict[str, Any],
    queries: Dict[str, str],
    qrels: Dict[str, Dict[str, int]],
    max_queries: int = 100,
    k_vals: List[int] = [1, 5, 10]
) -> pd.DataFrame:
    """Runs evaluation across all retrievers for a given benchmark."""
    test_qids = [qid for qid in queries.keys() if qid in qrels][:max_queries]
    print(f"\nEvaluating {benchmark_name} on {len(test_qids)} queries across {len(retrievers)} systems...")
    
    results = {}
    for name in retrievers.keys():
        results[name] = {
            **{f"Hit@{k}": [] for k in k_vals},
            **{f"MRR@{k}": [] for k in k_vals},
            **{f"nDCG@{k}": [] for k in k_vals}
        }
        
    for qid in test_qids:
        query_text = queries[qid]
        gt = qrels[qid]
        
        for name, ret_fn in retrievers.items():
            top_res = ret_fn(query_text, top_k=max(k_vals))
            retrieved_ids = [did for did, _ in top_res]
            
            for k in k_vals:
                results[name][f"Hit@{k}"].append(hit_rate_at_k(retrieved_ids, gt, k))
                results[name][f"MRR@{k}"].append(mrr_at_k(retrieved_ids, gt, k))
                results[name][f"nDCG@{k}"].append(ndcg_at_k(retrieved_ids, gt, k))
                
    summary = {}
    for name, metric_dict in results.items():
        summary[name] = {m: float(np.mean(vals)) for m, vals in metric_dict.items()}
        
    df = pd.DataFrame(summary).T
    ordered_cols = [f"{m}@{k}" for k in k_vals for m in ["Hit", "MRR", "nDCG"]]
    return df[[c for c in ordered_cols if c in df.columns]]

## 🏁 6. Comprehensive Benchmark Execution & Comparative Analysis

We execute all 4 retrieval strategies across both benchmarks:
- **SciFact** (Scientific exact-term domain)
- **NFCorpus** (Medical lay-question domain)

*Note: `max_queries=100` allows this cell to finish in under 30 seconds. Set `max_queries=len(queries)` to run on the full test sets.*

In [7]:
# Cell 7: Execute Benchmarks & Compare Results

# --------------------------------------------------------------------------
# 1. Benchmark 1: SciFact (Lexical / Exact-term Bias)
# --------------------------------------------------------------------------
print("=== BENCHMARK 1: SCIFACT ===")
bm25_scifact = BM25Retriever(scifact_corpus)
minilm_scifact = DenseRetriever(scifact_corpus, model_minilm, "MiniLM")
bge_scifact = DenseRetriever(scifact_corpus, model_bge, "BGE-small", query_prefix=BGE_QUERY_PREFIX)

scifact_systems = {
    "BM25": lambda q, top_k: bm25_scifact.retrieve(q, top_k=top_k),
    "Dense (MiniLM)": lambda q, top_k: minilm_scifact.retrieve(q, top_k=top_k),
    "Dense (BGE-small)": lambda q, top_k: bge_scifact.retrieve(q, top_k=top_k),
    "Hybrid (BM25 + BGE RRF)": lambda q, top_k: reciprocal_rank_fusion(
        [bm25_scifact.retrieve(q, top_k=100), bge_scifact.retrieve(q, top_k=100)],
        k=60,
        top_k=top_k
    )
}

df_scifact = evaluate_dataset(
    "SciFact", scifact_systems, scifact_queries, scifact_qrels, max_queries=100, k_vals=[1, 5, 10]
)
print("\n--- SciFact Results (Max 100 Test Queries) ---")
pd.set_option("display.precision", 4)
display(df_scifact)

# --------------------------------------------------------------------------
# 2. Benchmark 2: NFCorpus (Semantic Paraphrase / Layman-to-Medical Bias)
# --------------------------------------------------------------------------
print("\n=== BENCHMARK 2: NFCORPUS ===")
bm25_nfcorpus = BM25Retriever(nfcorpus_corpus)
minilm_nfcorpus = DenseRetriever(nfcorpus_corpus, model_minilm, "MiniLM")
bge_nfcorpus = DenseRetriever(nfcorpus_corpus, model_bge, "BGE-small", query_prefix=BGE_QUERY_PREFIX)

nfcorpus_systems = {
    "BM25": lambda q, top_k: bm25_nfcorpus.retrieve(q, top_k=top_k),
    "Dense (MiniLM)": lambda q, top_k: minilm_nfcorpus.retrieve(q, top_k=top_k),
    "Dense (BGE-small)": lambda q, top_k: bge_nfcorpus.retrieve(q, top_k=top_k),
    "Hybrid (BM25 + BGE RRF)": lambda q, top_k: reciprocal_rank_fusion(
        [bm25_nfcorpus.retrieve(q, top_k=100), bge_nfcorpus.retrieve(q, top_k=100)],
        k=60,
        top_k=top_k
    )
}

df_nfcorpus = evaluate_dataset(
    "NFCorpus", nfcorpus_systems, nfcorpus_queries, nfcorpus_qrels, max_queries=100, k_vals=[1, 5, 10]
)
print("\n--- NFCorpus Results (Max 100 Test Queries) ---")
display(df_nfcorpus)

=== BENCHMARK 1: SCIFACT ===
Building BM25 index over 5,183 passages...
BM25 index built in 0.77s
Pre-encoding 5,183 passages with MiniLM on cuda...
-> Encoded in 29.85s | Matrix shape: (5183, 384)
Pre-encoding 5,183 passages with BGE-small on cuda...
-> Encoded in 121.69s | Matrix shape: (5183, 384)

Evaluating SciFact on 100 queries across 4 systems...

--- SciFact Results (Max 100 Test Queries) ---


,Hit@1,MRR@1,nDCG@1,Hit@5,MRR@5,nDCG@5,Hit@10,MRR@10,nDCG@10
BM25,0.59,0.59,0.59,0.85,0.6905,0.7135,0.90,0.6979,0.7290
Dense (MiniLM),0.57,0.57,0.57,0.80,0.6632,0.6882,0.83,0.6673,0.6996
Dense (BGE-small),0.65,0.65,0.65,0.80,0.7085,0.7132,0.87,0.7182,0.7421
Hybrid (BM25 + BGE RRF),0.71,0.71,0.71,0.84,0.7585,0.7641,0.90,0.7672,0.7852



=== BENCHMARK 2: NFCORPUS ===
Building BM25 index over 3,633 passages...
BM25 index built in 0.35s
Pre-encoding 3,633 passages with MiniLM on cuda...
-> Encoded in 24.47s | Matrix shape: (3633, 384)
Pre-encoding 3,633 passages with BGE-small on cuda...
-> Encoded in 107.58s | Matrix shape: (3633, 384)

Evaluating NFCorpus on 100 queries across 4 systems...

--- NFCorpus Results (Max 100 Test Queries) ---


,Hit@1,MRR@1,nDCG@1,Hit@5,MRR@5,nDCG@5,Hit@10,MRR@10,nDCG@10
BM25,0.46,0.46,0.4333,0.66,0.5340,0.3628,0.74,0.5436,0.3396
Dense (MiniLM),0.48,0.48,0.4333,0.70,0.5552,0.3664,0.74,0.5611,0.3530
Dense (BGE-small),0.54,0.54,0.5133,0.72,0.6125,0.4308,0.73,0.6139,0.3987
Hybrid (BM25 + BGE RRF),0.54,0.54,0.5000,0.70,0.6023,0.4026,0.74,0.6069,0.3762


## 🔬 Key Empirical Observations & Takeaways

### 1. SciFact (Domain-Specific Exact Term Matching):
- **BM25's Resurgence**: In scientific verification with complex genes, molecules, and chemical abbreviations, BM25 performs remarkably well because exact terminology discriminates relevant documents unambiguously.
- **Dense Dilution**: Unspecialized bi-encoders (`all-MiniLM-L6-v2`) frequently lose exact entity precision in scientific corpora.

### 2. NFCorpus (Layman Vocabulary Mismatch):
- **BM25's Degradation**: BM25 suffers significantly because queries use colloquial phrases ("how to treat acne") while the target literature uses clinical medical terms ("isotretinoin efficacy in vulgaris vulgaris").
- **Dense Superiority**: Dense models (especially `bge-small-en-v1.5` with asymmetric search instructions) dominate BM25 by projecting colloquial questions and clinical abstracts into adjacent latent semantic space.

### 3. The Hybrid RRF Winner:
- **Universal Robustness**: Hybrid RRF is never caught in the catastrophic failure modes of either lexical or dense models alone. On both datasets, **RRF consistently matches or beats the best individual retriever** on `nDCG@10` without requiring score threshold tuning.